In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')
!pip install unsloth
!pip install snac
!pip install torchaudio
!pip install datasets transformers accelerate

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os
# DRIVE = "/content/drive/MyDrive/rio"
# os.makedirs(DRIVE, exist_ok=True)
# print("Drive folder:", DRIVE)
# print("Contents:", os.listdir(DRIVE))

import os

DRIVE = "/kaggle/input/datasets/lokeshk431/voices"  # or absolute path like "E:/something_new/rio"
os.makedirs(DRIVE, exist_ok=True)
print("Drive folder:", DRIVE)
print("Contents:", os.listdir(DRIVE))

In [ ]:

import json, os, torch, torchaudio
from snac import SNAC
from pathlib import Path
from tqdm import tqdm

INPUT_DIR   = "/kaggle/input/datasets/lokeshk431/voices"
WORK_DIR    = "/kaggle/working"
MAX_AUDIO_S = 8

device = "cuda" if torch.cuda.is_available() else "cpu"
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().to(device)

def audio_to_tokens(wav_path):
    waveform, sr = torchaudio.load(wav_path)
    if sr != 24000:
        waveform = torchaudio.functional.resample(waveform, sr, 24000)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(0, keepdim=True)
    if waveform.shape[-1] / 24000 > MAX_AUDIO_S:
        return None
    with torch.no_grad():
        codes = snac_model.encode(waveform.unsqueeze(0).to(device))
    c0 = codes[0].squeeze().cpu().tolist()
    c1 = codes[1].squeeze().cpu().tolist()
    c2 = codes[2].squeeze().cpu().tolist()
    if len(c1) < 2 * len(c0) or len(c2) < 4 * len(c0):
        return None
    tokens = []
    for i in range(len(c0)):
        tokens.append(c0[i])
        tokens.append(c1[2*i]     + 4096)
        tokens.append(c2[4*i]     + 8192)
        tokens.append(c2[4*i+1]   + 12288)
        tokens.append(c1[2*i+1]   + 16384)
        tokens.append(c2[4*i+2]   + 20480)
        tokens.append(c2[4*i+3]   + 24576)
    return tokens

jsonl_path = f"{INPUT_DIR}/dataset.jsonl"
wavs_path  = INPUT_DIR  # files uploaded flat, no subfolder

print(f"jsonl: {jsonl_path}")
print(f"wavs:  {wavs_path}")
print(f"wav files found: {len([f for f in os.listdir(wavs_path) if f.endswith('.wav')])}")

with open(jsonl_path) as f:
    samples = [json.loads(l) for l in f if l.strip()]

tokenized, skipped = [], 0
for item in tqdm(samples, desc="SNAC encoding"):
    filename = os.path.basename(item["audio_path"].replace("\\", "/"))
    wav_path = os.path.join(wavs_path, filename)
    if not os.path.exists(wav_path):
        skipped += 1
        continue
    try:
        tokens = audio_to_tokens(wav_path)
        if tokens is None or len(tokens) < 14:
            skipped += 1
            continue
        tokenized.append({"text": item["text"], "tokens": tokens})
    except Exception as e:
        print(f"  {filename}: {e}")
        skipped += 1

print(f"Encoded: {len(tokenized)}  |  skipped: {skipped}")

tok_path = f"{WORK_DIR}/tokenized.jsonl"
with open(tok_path, "w") as f:
    for item in tokenized:
        f.write(json.dumps(item) + "\n")
print("Saved:", tok_path)

In [ ]:
import json, os
from datasets import Dataset

INPUT_DIR   = "/kaggle/input/datasets/lokeshk431/voices"
WORK_DIR    = "/kaggle/working"
SPEAKER_NAME = "Aswini"

def format_sample(text, tokens):
    audio_str = "".join(f"<custom_token_{t + 10}>" for t in tokens)
    return (
        f"<|im_start|>user\n"
        f"<custom_token_3>{SPEAKER_NAME}: {text}<|eot_id|><custom_token_4>\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"<custom_token_5>{audio_str}<|im_end|>"
    )

tok_path = f"{WORK_DIR}/tokenized.jsonl"
if not os.path.exists(tok_path):
    raise FileNotFoundError(f"Tokenized file not found at {tok_path}. Please run the encoding cell first.")

with open(tok_path) as f:
    samples = [json.loads(l) for l in f if l.strip()]

if len(samples) == 0:
    print("Error: No samples found in tokenized.jsonl. Check cell 3f05cf12 output.")
else:
    formatted = [{"text": format_sample(s["text"], s["tokens"])} for s in samples]
    print(f"Total: {len(formatted)} samples")
    print("\nPreview (first 300 chars):")
    print(formatted[0]["text"][:300])
    print(f"\nCustom tokens in sample: {formatted[0]['text'].count('<custom_token_')}")

    split = int(len(formatted) * 0.9)
    train_dataset = Dataset.from_list(formatted[:split])
    val_dataset   = Dataset.from_list(formatted[split:])
    print(f"\nTrain: {len(train_dataset)}  |  Val: {len(val_dataset)}")

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="canopylabs/orpheus-3b-0.1-ft",
    max_seq_length=8192,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0, #0.05
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        output_dir="/content/checkpoints",
        num_train_epochs=30,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        warmup_ratio=0.1,
        lr_scheduler_type="cosine",
        logging_steps=10,
        eval_steps=50,
        save_steps=200,
        save_total_limit=2,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        dataset_text_field="text",
        max_seq_length=8192,
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("Done. Final loss above.")



In [ ]:
SAVE_PATH = "/kaggle/working/orpheus_lora_v2"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"LoRA adapter saved to: {SAVE_PATH}")

In [ ]:
merged_path = "/kaggle/working/orpheus_merged_v2"
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")

# Then convert to GGUF Q4 (4-bit quantized — runs on CPU, ~2GB file)
GGUF_SAVE_PATH = "/kaggle/working/orpheus_aswini_q4.gguf"
model.save_pretrained_gguf(
    GGUF_SAVE_PATH,
    tokenizer,
    quantization_method="q4_k_m"   # 4-bit, good quality-size tradeoff
)
print(f"GGUF saved: {GGUF_SAVE_PATH}")

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo("LokeshK431/orpheus-aswini", repo_type="model", token="HUGGINGFACE_TOKEN_PLACEHOLDER", exist_ok=True)
api.upload_file(
    path_or_fileobj="/tmp/orpheus_aswini_q4.gguf",
    path_in_repo="orpheus_aswini_q4.gguf",
    repo_id="lokeshk431/orpheus-aswini",
    repo_type="model",
    token="HUGGINGFACE_TOKEN_PLACEHOLDER"
)

In [ ]:
!pip -q install pyannote.audio pydub pandas tqdm

In [ ]:
# CELL 2 — Paths + HF Token
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

HF_TOKEN   = UserSecretsClient().get_secret('HF_TOKEN')

# Dataset path on Kaggle filesystem (NOT the URL path)
# URL: kaggle.com/datasets/lokeshk431/calles → filesystem: /kaggle/input/calles/
INPUT_DIR  = Path('/kaggle/input/datasets/lokeshk431/calles')
WORK_DIR   = Path('/kaggle/working')
WAV_DIR    = WORK_DIR / 'wavs_converted'
OUT_DIR    = WORK_DIR / 'aswini_clips'

WAV_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIO_EXT = {'.mpeg', '.mpg', '.wav', '.ogg', '.opus', '.m4a', '.mp3', '.flac'}
audio_files = sorted(p for p in INPUT_DIR.rglob('*') if p.suffix.lower() in AUDIO_EXT)

print(f'Found {len(audio_files)} audio files:')
for p in audio_files:
    print(' ', p.name)

In [ ]:
def ffprobe_audio(path: Path) -> dict:
    cmd = [
        'ffprobe', '-v', 'error',
        '-select_streams', 'a:0',
        '-show_entries', 'stream=codec_name,channels,channel_layout,sample_rate,duration',
        '-of', 'json',
        str(path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    data = json.loads(result.stdout)
    return data.get('streams', [{}])[0]

probe_rows = []
for path in audio_files:
    info = ffprobe_audio(path)
    probe_rows.append({
        'file': path.name,
        'channels': info.get('channels'),
        'layout': info.get('channel_layout'),
        'sample_rate': info.get('sample_rate'),
        'duration': info.get('duration'),
        'codec': info.get('codec_name'),
    })

probe_df = pd.DataFrame(probe_rows)
probe_df.to_csv(WORK_DIR / 'audio_probe.csv', index=False)
probe_df

In [ ]:
def extract_channel(input_path: Path, channel: str, output_path: Path):
    """channel must be 'left' or 'right'."""
    pan = 'c0=c0' if channel == 'left' else 'c0=c1'
    cmd = [
        'ffmpeg', '-y', '-i', str(input_path),
        '-af', f'pan=mono|{pan}',
        '-ar', '22050',
        str(output_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
import torch
from pyannote.audio import Pipeline

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    raise RuntimeError('Missing HF_TOKEN. Add it as a Kaggle secret or environment variable.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
pipeline = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token=HF_TOKEN)
pipeline.to(device)
print(f'Using device: {device}')

In [ ]:
def diarize_file(path: Path) -> pd.DataFrame:
    diarization = pipeline(str(path))
    rows = []
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        rows.append({
            'file': path.name,
            'speaker': speaker,
            'start': float(turn.start),
            'end': float(turn.end),
            'duration': float(turn.end - turn.start),
        })
    return pd.DataFrame(rows)

def save_speaker_previews(path: Path, segments: pd.DataFrame, seconds_per_speaker: int = 20):
    audio = AudioSegment.from_file(path)
    for speaker, speaker_segments in segments.groupby('speaker'):
        preview = AudioSegment.empty()
        used_ms = 0
        for row in speaker_segments.sort_values('duration', ascending=False).itertuples():
            start_ms = max(0, int(row.start * 1000))
            end_ms = min(len(audio), int(row.end * 1000))
            clip = audio[start_ms:end_ms]
            preview += clip
            used_ms += len(clip)
            if used_ms >= seconds_per_speaker * 1000:
                break
        out = PREVIEW_DIR / f'{path.stem}__{speaker}.wav'
        preview.export(out, format='wav')

# Start with one file so you can inspect speaker labels before processing everything.
sample_file = audio_files[0]
sample_segments = diarize_file(sample_file)
sample_segments.to_csv(SEGMENT_DIR / f'{sample_file.stem}.csv', index=False)
save_speaker_previews(sample_file, sample_segments)
sample_segments.groupby('speaker')['duration'].sum().sort_values(ascending=False)

In [ ]:
# For a single-file test, set this after listening to the preview clips.
TARGET_SPEAKER_BY_FILE = {
    sample_file.name: 'SPEAKER_00',
}

# Export mode: 'keep' means only target speaker. 'remove' means delete target speaker parts.
MODE = 'keep'

# If MODE == 'remove': True keeps original duration by replacing target segments with silence.
PRESERVE_TIMING_WITH_SILENCE = False

# Padding helps avoid clipped syllables at segment edges.
PADDING_SEC = 0.15
MIN_SEGMENT_SEC = 0.25

In [ ]:
def normalized_intervals(segments: pd.DataFrame, target_speaker: str, audio_ms: int) -> list[tuple[int, int]]:
    intervals = []
    for row in segments[segments['speaker'] == target_speaker].itertuples():
        if row.duration < MIN_SEGMENT_SEC:
            continue
        start_ms = max(0, int((row.start - PADDING_SEC) * 1000))
        end_ms = min(audio_ms, int((row.end + PADDING_SEC) * 1000))
        if end_ms > start_ms:
            intervals.append((start_ms, end_ms))

    if not intervals:
        return []

    intervals.sort()
    merged = [intervals[0]]
    for start, end in intervals[1:]:
        prev_start, prev_end = merged[-1]
        if start <= prev_end:
            merged[-1] = (prev_start, max(prev_end, end))
        else:
            merged.append((start, end))
    return merged

def export_trimmed(path: Path, segments: pd.DataFrame, target_speaker: str, mode: str):
    audio = AudioSegment.from_file(path)
    intervals = normalized_intervals(segments, target_speaker, len(audio))

    if mode == 'keep':
        output = AudioSegment.empty()
        for start_ms, end_ms in intervals:
            output += audio[start_ms:end_ms]
    elif mode == 'remove':
        output = AudioSegment.empty()
        cursor = 0
        for start_ms, end_ms in intervals:
            output += audio[cursor:start_ms]
            if PRESERVE_TIMING_WITH_SILENCE:
                output += AudioSegment.silent(duration=end_ms - start_ms, frame_rate=audio.frame_rate)
            cursor = end_ms
        output += audio[cursor:]
    else:
        raise ValueError("mode must be 'keep' or 'remove'")

    out = OUTPUT_DIR / f'{path.stem}__{mode}_{target_speaker}.wav'
    output.export(out, format='wav')
    return out

export_trimmed(sample_file, sample_segments, TARGET_SPEAKER_BY_FILE[sample_file.name], MODE)

In [ ]:
all_segments = []

for path in tqdm(audio_files):
    csv_path = SEGMENT_DIR / f'{path.stem}.csv'
    if csv_path.exists():
        segments = pd.read_csv(csv_path)
    else:
        segments = diarize_file(path)
        segments.to_csv(csv_path, index=False)
        save_speaker_previews(path, segments)
    all_segments.append(segments)

all_segments_df = pd.concat(all_segments, ignore_index=True)
all_segments_df.to_csv(WORK_DIR / 'all_segments.csv', index=False)

summary = all_segments_df.groupby(['file', 'speaker'])['duration'].sum().reset_index()
summary.to_csv(WORK_DIR / 'speaker_duration_summary.csv', index=False)
summary

In [ ]:
# Fill this after listening to preview clips for each file.
# Example:
# TARGET_SPEAKER_BY_FILE = {
#     'amarnath - aswini.mpeg': 'SPEAKER_01',
#     'anup menon - aswini.mpeg': 'SPEAKER_00',
# }

missing = [path.name for path in audio_files if path.name not in TARGET_SPEAKER_BY_FILE]
if missing:
    print('Missing target speaker labels for:')
    for name in missing:
        print(' -', name)
else:
    exported = []
    for path in tqdm(audio_files):
        segments = pd.read_csv(SEGMENT_DIR / f'{path.stem}.csv')
        exported.append(export_trimmed(path, segments, TARGET_SPEAKER_BY_FILE[path.name], MODE))
    print('Exported files:')
    for path in exported:
        print(path)

In [ ]:
!cd /kaggle/working && zip -qr speaker_trim_results.zip speaker_trim
print('/kaggle/working/speaker_trim_results.zip')

In [ ]:
# CELL 3 — Convert all MPEG → WAV (16kHz mono, required by pyannote)
import subprocess
from tqdm.auto import tqdm

converted = []
for src in tqdm(audio_files, desc='Converting'):
    dst = WAV_DIR / (src.stem + '.wav')
    if not dst.exists():
        subprocess.run(
            ['ffmpeg', '-y', '-i', str(src), '-ar', '16000', '-ac', '1', str(dst)],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
    if dst.exists():
        converted.append(dst)
    else:
        print(f'  FAILED: {src.name}')

print(f'Converted: {len(converted)} files')

In [ ]:
# CELL 4 — Load pyannote diarization pipeline
# Requires accepting terms at:
#   https://huggingface.co/pyannote/speaker-diarization-3.1
#   https://huggingface.co/pyannote/segmentation-3.0
from pyannote.audio import Pipeline
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pipeline = Pipeline.from_pretrained(
    'pyannote/speaker-diarization-3.1',
    token=HF_TOKEN
)
pipeline = pipeline.to(device)
print('Pipeline ready on', device)

In [ ]:
# CELL 5 — Diarize all calls and identify Aswini
# Strategy: Aswini is the OUTBOUND caller.
#   - She speaks first AND has the most total speaking time.
#   - We pick the speaker with the longest cumulative duration.
import json
from tqdm.auto import tqdm

MIN_CLIP_S = 2.5
MAX_CLIP_S = 8.0

all_segments = []  # will hold (wav_stem, start, end, text_placeholder)
diarization_log = {}

def get_annotation(diarization_result):
      """Return a pyannote Annotation across pyannote API versions."""
      if hasattr(diarization_result, "itertracks"):
          return diarization_result

      for attr in ["speaker_diarization", "diarization", "annotation"]:
          value = getattr(diarization_result, attr, None)
          if hasattr(value, "itertracks"):
              return value

      if isinstance(diarization_result, dict):
          for key in ["speaker_diarization", "diarization", "annotation"]:
              value = diarization_result.get(key)
              if hasattr(value, "itertracks"):
                  return value

      raise TypeError(f"Unsupported diarization output type: {type(diarization_result)}")

for wav in tqdm(converted, desc='Diarizing'):
    try:
        result = pipeline(str(wav))
        diarization = get_annotation(result)
        
    except Exception as e:
        print(f'  SKIP {wav.name}: {e}')
        continue

    # Accumulate duration per speaker
    speaker_dur = {}
    segments_by_speaker = {}
    for turn, _, speaker in diarization.itertracks(yield_label=True):
        dur = turn.end - turn.start
        speaker_dur[speaker] = speaker_dur.get(speaker, 0) + dur
        segments_by_speaker.setdefault(speaker, []).append((turn.start, turn.end))

    if not speaker_dur:
        print(f'  No speech detected: {wav.name}')
        continue

    # Aswini = speaker with MOST total speaking time
    aswini_label = max(speaker_dur, key=speaker_dur.get)
    aswini_dur   = speaker_dur[aswini_label]
    total_dur    = sum(speaker_dur.values())

    print(f'  {wav.name}: {len(speaker_dur)} speakers, '
          f'Aswini={aswini_label} ({aswini_dur:.1f}s / {total_dur:.1f}s total)')

    diarization_log[wav.stem] = {
        'aswini_label': aswini_label,
        'speaker_durations': speaker_dur
    }

    for start, end in segments_by_speaker[aswini_label]:
        dur = end - start
        if MIN_CLIP_S <= dur <= MAX_CLIP_S:
            all_segments.append({
                'source': wav.stem,
                'start': round(start, 3),
                'end':   round(end,   3),
                'duration': round(dur, 3)
            })

print(f'\nTotal Aswini segments ({MIN_CLIP_S}-{MAX_CLIP_S}s): {len(all_segments)}')

# Save diarization log
with open(WORK_DIR / 'diarization_log.json', 'w') as f:
    json.dump(diarization_log, f, indent=2)
result = pipeline(str(sample_file))
print(type(result))
print(dir(result))

In [ ]:
# CELL 6 — Export Aswini clips as WAV files
from pydub import AudioSegment as AS
from tqdm.auto import tqdm

PADDING_MS = 100  # add 100ms silence padding each side
wav_cache  = {}

exported = []
for i, seg in enumerate(tqdm(all_segments, desc='Exporting')):
    src_path = WAV_DIR / (seg['source'] + '.wav')
    if str(src_path) not in wav_cache:
        wav_cache[str(src_path)] = AS.from_wav(str(src_path))
    audio = wav_cache[str(src_path)]

    start_ms = max(0, int(seg['start'] * 1000) - PADDING_MS)
    end_ms   = min(len(audio), int(seg['end']   * 1000) + PADDING_MS)
    clip     = audio[start_ms:end_ms]

    out_name = f'aswini_{i:04d}.wav'
    out_path = OUT_DIR / out_name
    clip.export(str(out_path), format='wav')

    exported.append({
        'audio_path': str(out_path),
        'source': seg['source'],
        'start': seg['start'],
        'end': seg['end'],
        'duration': seg['duration'],
        'text': ''  # filled by Cell 7
    })

print(f'Exported {len(exported)} clips to {OUT_DIR}')

In [ ]:
# CELL 7 — Transcribe Aswini clips with Whisper
!pip -q install -U openai-whisper
import whisper
from tqdm.auto import tqdm

model = whisper.load_model('large-v2')  # use 'small' if this is slow
print('Whisper large-v2 loaded')

for item in tqdm(exported, desc='Transcribing'):
    try:
        result = model.transcribe(
            item['audio_path'],
            language='en',
            condition_on_previous_text=False
        )
        item['text'] = result['text'].strip()
    except Exception as e:
        item['text'] = ''
        print(f"  {item['audio_path']}: {e}")

# Drop empty transcripts
exported = [x for x in exported if x['text']]
print(f'With transcripts: {len(exported)}')

In [ ]:
# CELL 8 — Save dataset.jsonl + summary
import json

jsonl_path = WORK_DIR / 'aswini_dataset.jsonl'
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for item in exported:
        f.write(json.dumps(item) + '\n')

total_sec = sum(x['duration'] for x in exported)
print(f'Dataset saved: {jsonl_path}')
print(f'Clips  : {len(exported)}')
print(f'Total  : {total_sec/60:.1f} minutes of Aswini speech')
print()
print('--- Preview (first 5) ---')
for item in exported[:5]:
    print(f"  [{item['duration']:.1f}s] {item['text']}")
print()
print('Download from Kaggle Output tab:')
print('  /kaggle/working/aswini_clips/   (WAV files)')
print('  /kaggle/working/aswini_dataset.jsonl')

In [ ]:
!cd /kaggle/working && zip -qr aswini_speaker_dataset.zip \
 aswini_clips \
 aswini_dataset.jsonl \
 speaker_trim \
 diarization_log.json

In [ ]:
import shutil
from pathlib import Path

out = Path("/kaggle/working/aswini_download")
out.mkdir(exist_ok=True)

items = [
    "/kaggle/working/aswini_clips",
    "/kaggle/working/aswini_dataset.jsonl",
    "/kaggle/working/diarization_log.json",
    "/kaggle/working/speaker_trim",
]

for item in items:
    p = Path(item)
    if not p.exists():
        print("Missing, skipping:", p)
        continue

    dest = out / p.name
    if p.is_dir():
        shutil.copytree(p, dest, dirs_exist_ok=True)
    else:
        shutil.copy2(p, dest)
shutil.make_archive("/kaggle/working/aswini_speaker_dataset", "zip", out)
print("Download:", "/kaggle/working/aswini_speaker_dataset.zip")


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

zip_path = Path("/kaggle/working/aswini_speaker_dataset.zip")

print("Exists:", zip_path.exists())
print("Size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

display(FileLink(str(zip_path)))

In [ ]:
import shutil
from IPython.display import FileLink, display

src = "/kaggle/working/aswini_speaker_dataset.zip"
dst = "/kaggle/working/download.zip"

shutil.copy2(src, dst)
display(FileLink(dst))

In [ ]:
!cd /kaggle/working && split -b 100M aswini_speaker_dataset.zip aswini_part_
!ls -lh /kaggle/working/aswini_part_*

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

for p in sorted(Path("/kaggle/working").glob("aswini_part_*")):
    display(FileLink(str(p)))

In [ ]:
from IPython.display import FileLink, HTML, display
from pathlib import Path

p = Path("/kaggle/working/aswini_speaker_dataset.zip")

display(FileLink(p.name))
display(HTML(f'<a href="/files/kaggle/working/{p.name}" download>Download aswini_speaker_dataset.zip</a>'))

In [ ]:
%cd /kaggle/working
!ls -lh

In [ ]:
from IPython.display import FileLink, display
display(FileLink("aswini_speaker_dataset.zip"))

In [ ]:
!pip install -q faster-whisper soundfile pandas tqdm

In [ ]:
import json
import subprocess
import zipfile
from pathlib import Path

import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/my_voice_work")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".mpeg", ".mp4", ".flac", ".ogg"}


def guess_language(path):
    name = str(path).lower()
    if "hindi" in name:
        return "hindi"
    if "tamil" in name:
        return "tamil"
    if "bhojpuri" in name:
        return "bhojpuri"
    if "mixed" in name:
        return "mixed"
    return "english"


def get_duration_seconds(path):
    try:
        result = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                str(path),
            ],
            capture_output=True,
            text=True,
            check=True,
        )
        return round(float(result.stdout.strip()), 2)
    except Exception:
        return None


def build_manifest(audio_files):
    manifest = WORK_ROOT / "manifest.jsonl"

    with open(manifest, "w", encoding="utf-8") as f:
        for src in audio_files:
            row = {
                "audio_path": str(src),
                "speaker": "lk",
                "language": guess_language(src),
                "duration": get_duration_seconds(src),
                "text": "",
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    return manifest


def find_or_build_dataset():
    manifests = sorted(INPUT_ROOT.rglob("manifest.jsonl"))
    if manifests:
        manifest = manifests[0]
        return manifest, manifest.parent

    for z in sorted(INPUT_ROOT.rglob("*.zip")):
        extract_dir = WORK_ROOT / "uploaded_zip"
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(z, "r") as handle:
            handle.extractall(extract_dir)

        manifests = sorted(extract_dir.rglob("manifest.jsonl"))
        if manifests:
            manifest = manifests[0]
            return manifest, manifest.parent

        audio_files = sorted(
            p for p in extract_dir.rglob("*")
            if p.suffix.lower() in AUDIO_EXTS
        )
        if audio_files:
            return build_manifest(audio_files), WORK_ROOT

    audio_files = sorted(
        p for p in INPUT_ROOT.rglob("*")
        if p.suffix.lower() in AUDIO_EXTS
    )

    if not audio_files:
        raise FileNotFoundError("No audio files or manifest.jsonl found under /kaggle/input")

    return build_manifest(audio_files), WORK_ROOT


MANIFEST, DATA_ROOT = find_or_build_dataset()

rows = []
with open(MANIFEST, "r", encoding="utf-8-sig") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

print("Manifest:", MANIFEST)
print("Data root:", DATA_ROOT)
print("Rows:", len(df))

display(df[["speaker", "language", "duration", "audio_path"]].head())
display(df.groupby(["speaker", "language"])["duration"].agg(["count", "sum"]))


In [ ]:
# CELL 3 - Fix audio paths for Kaggle filesystem
def resolve_audio_path(row):
    original = Path(row['audio_path'])
    name = original.name
    candidates = list(DATA_ROOT.rglob(name))
    if candidates:
        return candidates[0]
    candidates = list(INPUT_ROOT.rglob(name))
    if candidates:
        return candidates[0]
    raise FileNotFoundError(f'Could not find audio file for {name}')

for row in rows:
    row['kaggle_audio_path'] = str(resolve_audio_path(row))

df = pd.DataFrame(rows)
display(df.groupby(['speaker', 'language'])['duration'].agg(['count', 'sum']))
print('Total minutes:', round(df['duration'].sum() / 60, 2))

In [ ]:
# CELL 3 - Install transcription tools
!pip install -q faster-whisper huggingface_hub

In [ ]:
 # Rebuild df from uploaded Kaggle audio files
from pathlib import Path
import subprocess
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".mpeg", ".mp4", ".flac", ".ogg"}

audio_files = sorted(
    p for p in INPUT_ROOT.rglob("*")
    if p.suffix.lower() in AUDIO_EXTS
)

print("Audio files found:", len(audio_files))
for p in audio_files:
    print(p)

def get_duration_seconds(path):
    try:
        result = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                str(path),
            ],
            capture_output=True,
            text=True,
            check=True,
        )
        return round(float(result.stdout.strip()), 2)
    except Exception:
        return None

df = pd.DataFrame([
    {
        "audio_path": str(p),
        "source_name": p.name,
        "speaker": "lk",
        "duration": get_duration_seconds(p),
    }
    for p in audio_files
])

display(df.head())
print("Rows:", len(df))

In [ ]:
from pathlib import Path

LANGUAGE_BY_FILE = {
    "expressive.wav": "english",
    "intro.wav": "english",
    "natural.wav": "english",
    "number.wav": "english",
    "paragraph.wav": "english",
    "start.wav": "english",
    "variation.wav": "english",

  "explain.wav": "hindi",
  "expressive1.wav": "hindi",
  "intro2.wav": "hindi",
  "purchase.wav": "hindi",
  "starting.wav": "hindi",

  "business_question.wav": "tamil",
  "emotion.wav": "tamil",
  "intro1.wav": "tamil",
  "paragraph1.wav": "tamil",

  "gags.wav": "bhojpuri",
  "intro3.wav": "bhojpuri",

  "mixed.wav": "mixed",
}

df["language"] = df["source_name"].map(LANGUAGE_BY_FILE).fillna("unknown")

display(df[["source_name", "language", "duration"]])
print(df["language"].value_counts())

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import json
import pandas as pd

TRANSCRIPT_DIR = Path("/kaggle/working/my_voice_work/transcripts")
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

model = WhisperModel("large-v3", device="cuda", compute_type="float16")

transcribed_rows = []

for i, row in df.iterrows():
    audio_path = Path(row["audio_path"])
    language = row["language"]

    # Whisper language codes
    lang_code = {
        "english": "en",
        "hindi": "hi",
        "tamil": "ta",
        "bhojpuri": "hi",   # fallback; Whisper has no strong Bhojpuri mode
        "mixed": None,
    }.get(language, None)

    print(f"[{i+1}/{len(df)}] Transcribing:", audio_path.name, "|", language)

    segments, info = model.transcribe(
        str(audio_path),
        language=lang_code,
        beam_size=5,
        vad_filter=True,
    )

    text_parts = []
    segment_rows = []

    for seg in segments:
        text = seg.text.strip()
        if not text:
            continue

        text_parts.append(text)
        segment_rows.append({
            "audio_path": str(audio_path),
            "speaker": row["speaker"],
            "language": language,
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "duration": round(seg.end - seg.start, 2),
            "text": text,
        })

    full_text = " ".join(text_parts).strip()

    out_txt = TRANSCRIPT_DIR / f"{audio_path.stem}.txt"
    out_txt.write_text(full_text, encoding="utf-8")

    new_row = dict(row)
    new_row["text"] = full_text
    new_row["detected_language"] = info.language
    new_row["language_probability"] = round(info.language_probability, 4)
    transcribed_rows.append(new_row)

    for seg_row in segment_rows:
        transcribed_rows.append(seg_row)

print("Done")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import json

TRANSCRIPT_DIR = Path("/kaggle/working/my_voice_transcripts_fixed")
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

model = WhisperModel("large-v3", device="cuda", compute_type="float16")

LANG_CODE = {
    "english": "en",
    "hindi": "hi",
    "tamil": "ta",
    "bhojpuri": "hi",   # fallback; Whisper does not properly support Bhojpuri
    "mixed": None,
}

rows_out = []

for i, row in df.iterrows():
    audio_path = Path(row["audio_path"])
    language = row["language"]
    lang_code = LANG_CODE.get(language)

    print(f"[{i+1}/{len(df)}] {audio_path.name} | {language} | whisper={lang_code}")

    segments, info = model.transcribe(
        str(audio_path),
        language=lang_code,
        task="transcribe",
        beam_size=5,
        vad_filter=True,
    )

    text_parts = []
    segment_rows = []

    for seg in segments:
        text = seg.text.strip()
        if not text:
            continue

        text_parts.append(text)
        segment_rows.append({
            "audio_path": str(audio_path),
            "source_name": audio_path.name,
            "speaker": "lk",
            "language": language,
            "detected_language": info.language,
            "start": round(seg.start, 2),
            "end": round(seg.end, 2),
            "duration": round(seg.end - seg.start, 2),
            "text": text,
        })

    full_text = " ".join(text_parts).strip()

    (TRANSCRIPT_DIR / f"{audio_path.stem}.txt").write_text(full_text, encoding="utf-8")

    rows_out.append({
        "audio_path": str(audio_path),
        "source_name": audio_path.name,
        "speaker": "lk",
        "language": language,
        "detected_language": info.language,
        "language_probability": round(info.language_probability, 4),
        "duration": row["duration"],
        "text": full_text,
    })

print("Done:", len(rows_out))

In [ ]:
from pathlib import Path

OUT_DIR = Path("/kaggle/working")
fixed_df = pd.DataFrame(rows_out)
fixed_df.to_csv(csv_path, index=False)

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in fixed_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(csv_path)
print(jsonl_path)

display(fixed_df[["source_name", "language", "detected_language", "language_probability", "text"]])

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

    print("HF_TOKEN loaded")
except Exception as e:
    HF_TOKEN = None
    print("HF_TOKEN not found, continuing unauthenticated:", e)

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

    print("HF_TOKEN loaded")
except Exception as e:
    HF_TOKEN = None
    print("HF_TOKEN not found, continuing unauthenticated:", e)
from huggingface_hub import login

if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)

  #Better Transcription Cell

  #This uses forced language for clean files and auto-detects only mixed.wav.

from faster_whisper import WhisperModel
from pathlib import Path
import json
import pandas as pd

TRANSCRIPT_DIR = Path("/kaggle/working/my_voice_transcripts_fixed")
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

model = WhisperModel(
    "large-v3",
    device="cuda",
    compute_type="float16",
)

LANG_CODE = {
    "english": "en",
    "hindi": "hi",
    "tamil": "ta",
    "telugu": "te",
    "bhojpuri": "hi",  # fallback; Whisper does not properly support Bhojpuri
    "mixed": None,     # auto-detect
}

rows_out = []

for i, row in df.iterrows():
    audio_path = Path(row["audio_path"])
    language = row["language"]
    lang_code = LANG_CODE.get(language)

    print(f"[{i+1}/{len(df)}] {audio_path.name} | label={language} | whisper={lang_code}")

    segments, info = model.transcribe(
        str(audio_path),
        language=lang_code,
        task="transcribe",
        beam_size=5,
        vad_filter=True,
        condition_on_previous_text=False,
    )

    text_parts = []

    for seg in segments:
        text = seg.text.strip()
        if text:
            text_parts.append(text)

    full_text = " ".join(text_parts).strip()

    (TRANSCRIPT_DIR / f"{audio_path.stem}.txt").write_text(full_text, encoding="utf-8")

    rows_out.append({
        "audio_path": str(audio_path),
        "source_name": audio_path.name,
        "speaker": "lk",
        "language": language,
        "whisper_language": lang_code or "auto",
        "detected_language": info.language,
        "language_probability": round(info.language_probability, 4),
        "duration": row["duration"],
        "text": full_text,
    })

fixed_df = pd.DataFrame(rows_out)

display(fixed_df[[
    "source_name",
    "language",
    "whisper_language",
    "detected_language",
    "language_probability",
    "text",
]])

  #Save Output

OUT_DIR = Path("/kaggle/working")

csv_path = OUT_DIR / "lk_transcripts_fixed.csv"
jsonl_path = OUT_DIR / "lk_transcripts_fixed.jsonl"

fixed_df.to_csv(csv_path, index=False)

with open(jsonl_path, "w", encoding="utf-8") as f:
    for row in fixed_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(csv_path)
print(jsonl_path)

In [ ]:
# CELL 5 - Save draft manifests
import json
import pandas as pd
from pathlib import Path

OUT_DIR = Path("/kaggle/working/my_voice_work")
OUT_DIR.mkdir(parents=True, exist_ok=True)

draft_df = pd.DataFrame(transcribed_rows)

draft_jsonl = OUT_DIR / "lk_transcribed_dataset_draft.jsonl"
draft_csv = OUT_DIR / "lk_transcribed_dataset_draft.csv"

with open(draft_jsonl, "w", encoding="utf-8") as f:
    for row in draft_df.to_dict(orient="records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

draft_df.to_csv(draft_csv, index=False)

print("Saved:", draft_jsonl)
print("Saved:", draft_csv)

display(draft_df.head())

In [ ]:
#Then CELL 6 to zip the result:
# CELL 6 - Zip outputs for download
!cd /kaggle/working && zip -qr lk_voice_transcripts.zip my_voice_work

print("/kaggle/working/lk_voice_transcripts.zip")

In [ ]:
# CELL 4 - Load faster-whisper model
from faster_whisper import WhisperModel

# Recommended:
# - base: fastest, lower quality
# - medium: good balance on Kaggle T4
# - large-v3: best quality, slower/heavier
MODEL_SIZE = 'large-v3'
DEVICE = 'cuda'
COMPUTE_TYPE = 'float16'

model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print('Loaded', MODEL_SIZE)

In [ ]:
# CELL 5 - Transcribe files
TRANSCRIPTS_DIR = Path('/kaggle/working/my_voice_transcripts')
TRANSCRIPTS_DIR.mkdir(parents=True, exist_ok=True)

LANG_HINTS = {
    'english': 'en',
    'hindi': 'hi',
    'tamil': 'ta',
    # Bhojpuri is not a first-class Whisper language. Hindi hint often works better than auto.
    'bhojpuri': 'hi',
    'mixed': None,
}

transcript_rows = []

for row in tqdm(rows):
    audio_path = Path(row['kaggle_audio_path'])
    stem = audio_path.stem
    lang = LANG_HINTS.get(row.get('language'), None)
    json_path = TRANSCRIPTS_DIR / f'{stem}.json'
    txt_path = TRANSCRIPTS_DIR / f'{stem}.txt'

    if json_path.exists():
        payload = json.loads(json_path.read_text(encoding='utf-8'))
    else:
        segments, info = model.transcribe(
            str(audio_path),
            language=lang,
            beam_size=5,
            vad_filter=True,
            vad_parameters=dict(min_silence_duration_ms=450),
        )
        segs = []
        for s in segments:
            segs.append({
                'start': round(float(s.start), 3),
                'end': round(float(s.end), 3),
                'text': s.text.strip(),
            })
        text = ' '.join(s['text'] for s in segs).strip()
        payload = {
            **row,
            'detected_language': getattr(info, 'language', None),
            'language_probability': float(getattr(info, 'language_probability', 0.0) or 0.0),
            'model': MODEL_SIZE,
            'segments': segs,
            'text': text,
        }
        json_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        txt_path.write_text(text + '\n', encoding='utf-8')

    transcript_rows.append({**row, 'transcript_json': str(json_path), 'transcript_txt': str(txt_path), 'draft_text': payload.get('text', '')})

transcript_manifest = Path('/kaggle/working/transcripts_manifest.jsonl')
with open(transcript_manifest, 'w', encoding='utf-8') as f:
    for row in transcript_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

print('Wrote', transcript_manifest)

In [ ]:
# CELL 6 - Build segmented draft TTS dataset
DATASET_DIR = Path('/kaggle/working/my_voice_dataset')
WAVS_DIR = DATASET_DIR / 'wavs'
WAVS_DIR.mkdir(parents=True, exist_ok=True)

MIN_SEC = 2.0
MAX_SEC = 14.0
MIN_CHARS = 8
PAD_SEC = 0.08

def ffmpeg_slice(src, dst, start, end):
    start = max(0.0, float(start) - PAD_SEC)
    duration = max(0.1, float(end) - start + PAD_SEC)
    cmd = [
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-ss', f'{start:.3f}', '-t', f'{duration:.3f}',
        '-i', str(src),
        '-ac', '1', '-ar', '24000', '-sample_fmt', 's16',
        str(dst),
    ]
    subprocess.run(cmd, check=True)

dataset_rows = []
counter = 0

for row in tqdm(transcript_rows):
    payload = json.loads(Path(row['transcript_json']).read_text(encoding='utf-8'))
    src = Path(row['kaggle_audio_path'])
    for seg in payload.get('segments', []):
        text = seg.get('text', '').strip()
        dur = float(seg['end']) - float(seg['start'])
        if dur < MIN_SEC or dur > MAX_SEC:
            continue
        if len(text) < MIN_CHARS:
            continue
        out_name = f"{row['speaker']}_{row['language']}_{counter:05d}.wav"
        out_path = WAVS_DIR / out_name
        ffmpeg_slice(src, out_path, seg['start'], seg['end'])
        dataset_rows.append({
            'audio_path': str(out_path),
            'text': text,
            'speaker': row['speaker'],
            'language': row['language'],
            'source_file': src.name,
            'start': seg['start'],
            'end': seg['end'],
            'duration': round(dur, 3),
            'needs_review': True,
        })
        counter += 1

dataset_jsonl = DATASET_DIR / 'dataset.draft.jsonl'
metadata_csv = DATASET_DIR / 'metadata.draft.csv'

with open(dataset_jsonl, 'w', encoding='utf-8') as f:
    for row in dataset_rows:
        f.write(json.dumps(row, ensure_ascii=False) + '\n')

pd.DataFrame(dataset_rows).to_csv(metadata_csv, index=False)

print('Segments:', len(dataset_rows))
print('Total minutes:', round(sum(r['duration'] for r in dataset_rows) / 60, 2))
print('Dataset:', DATASET_DIR)

In [ ]:
# CELL 7 - Preview summary
draft = pd.DataFrame(dataset_rows)
if len(draft):
    display(draft.groupby(['speaker', 'language'])['duration'].agg(['count', 'sum']))
    display(draft.sample(min(10, len(draft)), random_state=1)[['speaker', 'language', 'duration', 'text']])
else:
    print('No segments produced. Check transcription output and filters.')

In [ ]:
# CELL 8 - Zip outputs for download
!cd /kaggle/working && zip -qr my_voice_transcripts.zip my_voice_transcripts transcripts_manifest.jsonl
!cd /kaggle/working && zip -qr my_voice_dataset_draft.zip my_voice_dataset
print('/kaggle/working/my_voice_transcripts.zip')
print('/kaggle/working/my_voice_dataset_draft.zip')

In [ ]:
"""
Kaggle script: split lk_voice_dataset_fixed.zip into short TTS clips.

Assumed Kaggle input:
  /kaggle/input/datasets/lokeshk431/lkvoice/lk_voice_dataset_fixed.zip

Output:
  /kaggle/working/lk_voice_dataset_chunks/
    wavs/*.wav
    metadata.csv
    metadata.jsonl
    metadata_pipe.csv
    review.csv
    README.txt
  /kaggle/working/lk_voice_dataset_chunks.zip

Recommended Kaggle setup:
  Accelerator: GPU T4 if available, CPU also works but transcription is slower.
  Internet: On for first dependency/model download.

Run in a Kaggle notebook:
  !python /kaggle/working/kaggle_chunk_lk_voice_dataset.py
"""

from __future__ import annotations

import csv
import json
import math
import os
import re
import shutil
import subprocess
import sys
import wave
import zipfile
from pathlib import Path


INPUT_DATASET_DIR = Path("/kaggle/input/datasets/lokeshk431/lkvoice")
WORK_ROOT = Path("/kaggle/working/lk_voice_chunk_work")
EXTRACT_DIR = WORK_ROOT / "extracted"
OUT_DIR = Path("/kaggle/working/lk_voice_dataset_chunks")
OUT_WAVS = OUT_DIR / "wavs"
OUT_ZIP = Path("/kaggle/working/lk_voice_dataset_chunks.zip")

TARGET_SR = 24000
MIN_CHUNK_SEC = 3.0
MAX_CHUNK_SEC = 18.0
SOFT_CHUNK_SEC = 12.0
SILENCE_DB = -40
MIN_SILENCE_SEC = 0.35
PADDING_SEC = 0.08

LANGUAGE_HINTS = {
    "english": "en",
    "hindi": "hi",
    "tamil": "ta",
    # Bhojpuri support is weak in most ASR models; Hindi hint is usually better than auto.
    "bhojpuri": "hi",
}


def run(cmd: list[str]) -> str:
    result = subprocess.run(cmd, check=True, text=True, capture_output=True)
    return result.stdout.strip()


def ensure_clean_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def find_zip() -> Path | None:
    candidates = sorted(INPUT_DATASET_DIR.rglob("*.zip"))
    if not candidates:
        candidates = sorted(Path("/kaggle/input").rglob("lk_voice_dataset*.zip"))
    if not candidates:
        return None
    print("Using zip:", candidates[0])
    return candidates[0]


def extract_dataset() -> Path:
    ensure_clean_dir(EXTRACT_DIR)
    zip_path = find_zip()
    if zip_path is not None:
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(EXTRACT_DIR)
    else:
        print("No zip found. Looking for already-extracted metadata under /kaggle/input...")
        direct_metadata = sorted(INPUT_DATASET_DIR.rglob("metadata.jsonl"))
        if not direct_metadata:
            direct_metadata = sorted(INPUT_DATASET_DIR.rglob("metadata.csv"))
        if not direct_metadata:
            direct_metadata = sorted(Path("/kaggle/input").rglob("metadata.jsonl"))
        if not direct_metadata:
            direct_metadata = sorted(Path("/kaggle/input").rglob("metadata.csv"))
        if direct_metadata:
            print("Using extracted dataset metadata:", direct_metadata[0])
            return direct_metadata[0]
        raise FileNotFoundError(
            "Could not find a zip or extracted metadata.csv/metadata.jsonl under /kaggle/input. "
            "Run: !find /kaggle/input -maxdepth 5 -type f | sort | head -100"
        )
    metadata = sorted(EXTRACT_DIR.rglob("metadata.jsonl"))
    if metadata:
        return metadata[0]
    metadata_csv = sorted(EXTRACT_DIR.rglob("metadata.csv"))
    if metadata_csv:
        return metadata_csv[0]
    raise FileNotFoundError("No metadata.jsonl or metadata.csv found inside uploaded zip")


def load_metadata(path: Path) -> list[dict]:
    if path.suffix.lower() == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8-sig") as handle:
            for line in handle:
                if line.strip():
                    rows.append(json.loads(line))
        return rows
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


def source_wav(row: dict, metadata_path: Path) -> Path:
    rel = row["audio_file"]
    candidate = metadata_path.parent / rel
    if candidate.exists():
        return candidate
    matches = sorted(EXTRACT_DIR.rglob(Path(rel).name))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Missing wav for {rel}")


def wav_duration(path: Path) -> float:
    with wave.open(str(path), "rb") as handle:
        return handle.getnframes() / float(handle.getframerate())


def ffmpeg_convert(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    run(
        [
            "ffmpeg",
            "-y",
            "-hide_banner",
            "-loglevel",
            "error",
            "-i",
            str(src),
            "-ac",
            "1",
            "-ar",
            str(TARGET_SR),
            "-sample_fmt",
            "s16",
            str(dst),
        ]
    )


def detect_silences(path: Path) -> list[tuple[float, float]]:
    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-nostats",
        "-i",
        str(path),
        "-af",
        f"silencedetect=noise={SILENCE_DB}dB:d={MIN_SILENCE_SEC}",
        "-f",
        "null",
        "-",
    ]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    text = proc.stderr
    starts = [float(x) for x in re.findall(r"silence_start: ([0-9.]+)", text)]
    ends = [float(x) for x in re.findall(r"silence_end: ([0-9.]+)", text)]
    return list(zip(starts, ends))


def split_ranges(duration: float, silences: list[tuple[float, float]]) -> list[tuple[float, float]]:
    # Candidate breakpoints are silence midpoints. Build chunks up to SOFT_CHUNK_SEC,
    # but never longer than MAX_CHUNK_SEC unless there is no usable silence.
    breakpoints = [0.0]
    breakpoints.extend((start + end) / 2 for start, end in silences if 0 < start < duration)
    breakpoints.append(duration)
    breakpoints = sorted(set(round(x, 3) for x in breakpoints))

    ranges: list[tuple[float, float]] = []
    start = 0.0
    while start < duration - 0.1:
        target = min(start + SOFT_CHUNK_SEC, duration)
        hard = min(start + MAX_CHUNK_SEC, duration)
        candidates = [bp for bp in breakpoints if start + MIN_CHUNK_SEC <= bp <= hard]
        if candidates:
            before_target = [bp for bp in candidates if bp <= target]
            end = before_target[-1] if before_target else candidates[0]
        else:
            end = hard
        if end - start < MIN_CHUNK_SEC:
            break
        ranges.append((max(0.0, start - PADDING_SEC), min(duration, end + PADDING_SEC)))
        start = end
    return ranges


def slice_wav(src: Path, dst: Path, start: float, end: float) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    run(
        [
            "ffmpeg",
            "-y",
            "-hide_banner",
            "-loglevel",
            "error",
            "-ss",
            f"{start:.3f}",
            "-to",
            f"{end:.3f}",
            "-i",
            str(src),
            "-ac",
            "1",
            "-ar",
            str(TARGET_SR),
            "-sample_fmt",
            "s16",
            str(dst),
        ]
    )


def transcribe_chunks(rows: list[dict]) -> list[dict]:
    from faster_whisper import WhisperModel

    device = "cuda" if shutil.which("nvidia-smi") else "cpu"
    compute_type = "float16" if device == "cuda" else "int8"
    print(f"Loading faster-whisper small on {device}/{compute_type}")
    model = WhisperModel("small", device=device, compute_type=compute_type)

    output = []
    for index, row in enumerate(rows, start=1):
        language = row["language"]
        hint = LANGUAGE_HINTS.get(language)
        print(f"[{index}/{len(rows)}] ASR {row['audio_file']} lang={hint or 'auto'}")
        segments, info = model.transcribe(
            str(OUT_DIR / row["audio_file"]),
            language=hint,
            vad_filter=True,
            beam_size=5,
        )
        text = " ".join(seg.text.strip() for seg in segments if seg.text.strip())
        item = {
            **row,
            "text": " ".join(text.split()),
            "asr_language": getattr(info, "language", ""),
            "asr_language_probability": round(float(getattr(info, "language_probability", 0.0)), 4),
        }
        output.append(item)
    return output


def write_outputs(rows: list[dict], skipped: list[dict]) -> None:
    metadata_csv = OUT_DIR / "metadata.csv"
    metadata_jsonl = OUT_DIR / "metadata.jsonl"
    metadata_pipe = OUT_DIR / "metadata_pipe.csv"
    review_csv = OUT_DIR / "review.csv"

    fields = [
        "audio_file",
        "speaker",
        "language",
        "duration",
        "text",
        "source_audio_file",
        "source_start",
        "source_end",
        "source_text",
        "asr_language",
        "asr_language_probability",
    ]

    with metadata_csv.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)

    with metadata_jsonl.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

    with metadata_pipe.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter="|")
        for row in rows:
            writer.writerow([Path(row["audio_file"]).stem, row["text"]])

    review_fields = fields + ["keep", "notes"]
    with review_csv.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=review_fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({**row, "keep": "", "notes": ""})

    total_minutes = sum(float(row["duration"]) for row in rows) / 60.0
    by_language: dict[str, list[float]] = {}
    for row in rows:
        by_language.setdefault(row["language"], []).append(float(row["duration"]))

    readme = [
        "LK short-clip dataset generated on Kaggle",
        "",
        f"Samples: {len(rows)}",
        f"Total minutes: {total_minutes:.2f}",
        f"Target sample rate: {TARGET_SR}",
        f"Chunk length target: {MIN_CHUNK_SEC}-{MAX_CHUNK_SEC} sec",
        "",
        "By language:",
    ]
    for language, durations in sorted(by_language.items()):
        readme.append(f"- {language}: {len(durations)} clips, {sum(durations) / 60.0:.2f} min")
    readme.extend(
        [
            "",
            "Important:",
            "- Chunk text is generated by faster-whisper per clip.",
            "- Review review.csv before using this for final TTS fine-tuning.",
            "- Delete clips with wrong text, silence, repeated words, or bad pronunciation.",
        ]
    )
    if skipped:
        readme.extend(["", "Skipped:"])
        for item in skipped:
            readme.append(f"- {item}")
    (OUT_DIR / "README.txt").write_text("\n".join(readme) + "\n", encoding="utf-8")

    if OUT_ZIP.exists():
        OUT_ZIP.unlink()
    with zipfile.ZipFile(OUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUT_DIR.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(OUT_DIR.parent))


def main() -> None:
    ensure_clean_dir(WORK_ROOT)
    ensure_clean_dir(OUT_DIR)
    OUT_WAVS.mkdir(parents=True, exist_ok=True)

    metadata_path = extract_dataset()
    input_rows = load_metadata(metadata_path)
    print("Metadata:", metadata_path)
    print("Input rows:", len(input_rows))

    chunk_rows: list[dict] = []
    skipped: list[dict] = []

    converted_dir = WORK_ROOT / "converted"
    converted_dir.mkdir(parents=True, exist_ok=True)

    for row_index, row in enumerate(input_rows, start=1):
        try:
            src = source_wav(row, metadata_path)
            converted = converted_dir / src.name
            ffmpeg_convert(src, converted)
            duration = wav_duration(converted)
            silences = detect_silences(converted)
            ranges = split_ranges(duration, silences)
            print(f"[{row_index}/{len(input_rows)}] {src.name}: {duration:.1f}s -> {len(ranges)} chunks")

            for chunk_index, (start, end) in enumerate(ranges, start=1):
                out_name = f"{Path(row['audio_file']).stem}_{chunk_index:03d}.wav"
                out_path = OUT_WAVS / out_name
                slice_wav(converted, out_path, start, end)
                actual_duration = round(wav_duration(out_path), 2)
                if actual_duration < MIN_CHUNK_SEC:
                    out_path.unlink(missing_ok=True)
                    continue
                chunk_rows.append(
                    {
                        "audio_file": f"wavs/{out_name}",
                        "speaker": row.get("speaker", "lk"),
                        "language": row.get("language", ""),
                        "duration": actual_duration,
                        "text": "",
                        "source_audio_file": row.get("audio_file", src.name),
                        "source_start": round(start, 3),
                        "source_end": round(end, 3),
                        "source_text": row.get("text", ""),
                    }
                )
        except Exception as exc:
            skipped.append({"row": row_index, "audio_file": row.get("audio_file"), "error": str(exc)})
            print("SKIP:", skipped[-1])

    print("Chunks before ASR:", len(chunk_rows))
    chunk_rows = transcribe_chunks(chunk_rows)
    chunk_rows = [row for row in chunk_rows if row["text"].strip()]
    print("Chunks after ASR text filter:", len(chunk_rows))

    write_outputs(chunk_rows, skipped)
    print("Done")
    print("Output folder:", OUT_DIR)
    print("Output zip:", OUT_ZIP)


if __name__ == "__main__":
    main()


In [ ]:
from IPython.display import FileLink
FileLink('/kaggle/working/lk_voice_dataset_chunks.zip')

In [ ]:
import shutil
from pathlib import Path

src = Path("/kaggle/working/lk_voice_dataset_chunks")
dst = Path("/kaggle/working/lk_voice_dataset_chunks_download")

if dst.exists():
    shutil.rmtree(dst)

shutil.copytree(src, dst)
print("Copied to:", dst)
print("Files:", len(list(dst.rglob("*"))))

In [ ]:
!ls -lh /kaggle/working
!ls -lh /kaggle/working/lk_voice_dataset_chunks_download | head

In [ ]:
!ls -lh /kaggle/working/lk_voice_dataset_chunks.zip
!mkdir -p /kaggle/working/final_output
!cp /kaggle/working/lk_voice_dataset_chunks.zip /kaggle/working/final_output/
!cp -r /kaggle/working/lk_voice_dataset_chunks /kaggle/working/final_output/lk_voice_dataset_chunks_folder
!du -sh /kaggle/working/final_output